<a href="https://colab.research.google.com/github/OdysseusPolymetis/enexdi_prep_2026/blob/main/3_word_vectors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <center>**Word Vectors**</center>

---



##**Definition**

You can try and imagine language as a cloud, with scattered points, where each point is a different word. The location of each point is dependent on the location of every other point in the cloud (eg. if two words share the same context, they should appear near one to another). As long as you can represent a point in space, it gets a computational representation : it becomes a vector in space, a direction. And it becomes possible to compute things from it.

![](https://drive.google.com/uc?export=view&id=1FsTcOQ5LVgbDqkT5nm_gve5gZfQrZ8pV)

In [ ]:
!pip -q install stanza langdetect tqdm gensim langdetect

In [ ]:
import os
import gensim
from gensim.models import Word2Vec
import glob
import nltk

from lxml import etree as ET
import lxml.html
import string
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from google.colab import files
from pathlib import Path
import shutil
from langdetect import detect, DetectorFactory
import re
import stanza
from tqdm import tqdm

In [ ]:
!wget https://github.com/OdysseusPolymetis/enexdi_prep_2026/raw/refs/heads/main/auteurs.zip

In [ ]:
!unzip "/content/auteurs.zip"

In [ ]:
flaubert="auteurs/flaubert/"
balzac="auteurs/balzac/"

In [ ]:
def strip_ns_prefix(tree):
    query = "descendant-or-self::*[namespace-uri()!='']"
    for element in tree.xpath(query):
        element.tag = ET.QName(element).localname
    return tree

In [ ]:
if balzac != "":
    files = glob.iglob(balzac + '/**/*.xml', recursive=True)
    sentences = []

    for filename in files:
        print(filename)
        parser = ET.XMLParser(remove_blank_text=True, resolve_entities=False, encoding='utf8')
        tree = strip_ns_prefix(ET.parse(filename, parser))

        words = tree.xpath(".//wf/@lemma")

        sentence = []
        for word in words:
            if word != ".":
                sentence.append(word)
            else:
                sentences.append(sentence + [word])
                sentence = []

In [ ]:
print(len(sentences))
print(sentences[5])

## **Building a model**

This part, depending on the amount of data you intend to compute, may take some time (default : 8 minutes)

In [ ]:
model = Word2Vec(sentences, min_count=2, max_vocab_size=10000, negative=10, epochs=200)

In [ ]:
model.wv.save("/content/model_balzac.bin")

cellule suivante à n'utiliser que si vous rechargez le modèle

In [ ]:
from gensim.models import KeyedVectors
KeyedVectors.load("/content/model_balzac.bin")
wv = KeyedVectors.load("/content/model_balzac.bin")

model = Word2Vec(vector_size=wv.vector_size, min_count=1)
model.wv = wv

In [ ]:
print(model.wv.index_to_key)

In [ ]:
#Paris is to France what London is to what ? model.wv.most_similar(positive=['Londres', 'France'], negative=['Paris'],topn=5)
#King is to man what Queen is to what ? model.wv.most_similar(positive=['reine', 'homme'], negative=['roi'],topn=5)
model.wv.most_similar(positive=['reine', 'homme'], negative=['roi'],topn=10)

In [ ]:
model.wv.most_similar('père',topn=20)

## **With your own corpus**

For this experiment, your corpus will have to be in `txt` format, preferably in one file (but ok if you've got many). In the next cell, you'll be able to upload your own corpus. Sorry, I did the code in French, as I generally use it in regular classes.

In [ ]:
corpus_dir = Path("/content/my_own")

if corpus_dir.exists():
    shutil.rmtree(corpus_dir)

corpus_dir.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()

txt_paths = []

for filename, data in uploaded.items():
    filename_simple = Path(filename).name

    if not filename_simple.lower().endswith(".txt"):
        print(f"Fichier ignoré, car ce n'est pas un .txt : {filename_simple}")
        continue

    output_path = corpus_dir / filename_simple
    output_path.write_bytes(data)
    txt_paths.append(output_path)

print(f"{len(txt_paths)} fichier(s) .txt uploadé(s).")
for path in txt_paths:
    print("-", path)

This part reads the texts you uploaded. In unicode.

In [ ]:
def read_txt(path):
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin-1"]

    for enc in encodings:
        try:
            return path.read_text(encoding=enc), enc
        except UnicodeDecodeError:
            pass
    return path.read_text(encoding="utf-8", errors="replace"), "utf-8 avec remplacements"

texts = []

This part agglomerates your texts in one big fat one.

In [ ]:
for path in sorted(txt_paths):
    text, encoding_used = read_txt(path)
    print(f"{path.name} lu avec l'encodage : {encoding_used}")

corpus_brut = "\n".join(texts)

concat_path = Path("/content/corpus_concatene.txt")
concat_path.write_text(corpus_brut, encoding="utf-8")

print("Corpus concaténé enregistré dans :", concat_path)

This part detects the language it's written in.

In [ ]:
DetectorFactory.seed = 42

LANGUE_MANUELLE = None

sample = re.sub(r"\s+", " ", corpus_brut[:50000])

langue_detectee = detect(sample)
langue = LANGUE_MANUELLE if LANGUE_MANUELLE is not None else langue_detectee

print("Langue détectée :", langue_detectee)
print("Langue utilisée pour Stanza :", langue)

This part lemmatizes your text(s).

In [ ]:
try:
    import torch
    USE_GPU = torch.cuda.is_available()
except Exception:
    USE_GPU = False

print("GPU utilisé par Stanza :", USE_GPU)

def charger_pipeline_stanza(langue):
    essais_processors = [
        "tokenize,mwt,pos,lemma",
        "tokenize,pos,lemma"
    ]

    last_error = None

    for processors in essais_processors:
        try:
            print(f"Téléchargement/chargement Stanza avec : {processors}")
            stanza.download(langue, processors=processors, verbose=False)

            nlp = stanza.Pipeline(
                lang=langue,
                processors=processors,
                use_gpu=USE_GPU,
                verbose=False
            )

            print("Pipeline chargée :", processors)
            return nlp, processors

        except Exception as e:
            print(f"Échec avec {processors}")
            print(e)
            last_error = e

    raise RuntimeError(f"Impossible de charger une pipeline Stanza pour {langue}") from last_error

nlp, processors_utilises = charger_pipeline_stanza(langue)

And this part cuts your texts in bits so not to unload too much on stanza.

In [ ]:
def cut_blocks(text, max_chars=20000):
    paragraphes = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    blocs = []
    bloc_courant = []
    taille_courante = 0
    for p in paragraphes:
        while len(p) > max_chars:
            cut = p.rfind(" ", 0, max_chars)
            if cut == -1:
                cut = max_chars
            morceau = p[:cut].strip()
            if morceau:
                blocs.append(morceau)
            p = p[cut:].strip()

        if taille_courante + len(p) > max_chars and bloc_courant:
            blocs.append("\n\n".join(bloc_courant))
            bloc_courant = [p]
            taille_courante = len(p)
        else:
            bloc_courant.append(p)
            taille_courante += len(p)

    if bloc_courant:
        blocs.append("\n\n".join(bloc_courant))

    return blocs

blocs = cut_blocks(corpus_brut, max_chars=20000)

In [ ]:
sentences = []

for bloc in tqdm(blocs):
    doc = nlp(bloc)

    for sent in doc.sentences:
        phrase = [
            word.lemma.lower()
            for word in sent.words
            if word.upos != "PUNCT"
        ]

        if len(phrase) > 1:
            sentences.append(phrase)

print("Nombre de phrases :", len(sentences))
print(sentences[0])

And this is where the model works.

In [ ]:
model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=100
)

And finally you can ask questions.

In [ ]:
model.wv.most_similar("père", topn=20)

In [ ]:
model.wv.most_similar(positive=['reine', 'homme'], negative=['roi'],topn=10)

## **Visualization**

You can also get a much clearer visualization using the [online tensorflow visualizer](https://projector.tensorflow.org/). After this next cell, you'll get two files, one containing the vectors, the other their labels.

In [ ]:
vectors_path = "/content/vecteurs.tsv"
metadata_path = "/content/metadonnees.tsv"

with open(vectors_path, "w", encoding="utf-8") as file_vectors, \
     open(metadata_path, "w", encoding="utf-8") as file_metadata:

    for word in model.wv.index_to_key:
        vector = model.wv[word]
        file_vectors.write("\t".join(str(x) for x in vector) + "\n")
        file_metadata.write(word + "\n")

print("Fichiers Projector créés :")
print("-", vectors_path)
print("-", metadata_path)

In [ ]:
files.download("/content/vecteurs.tsv")
files.download("/content/metadonnees.tsv")